# Detección de audios naturales y sintéticos

El proyecto busca clasificar audios naturales y sintéticos mediante
la fusión de las puntuaciones generadas por cuatro modelos DenseNet121,
correspondientes a las representaciones LOG, MEL, DWT y CQT.

Cada audio se representará mediante cuatro puntuaciones de
fake_probability, conservadas en el rango de 0 a 100.

La etiqueta verdadera se codificará como:
- 0: audio natural (real).
- 1: audio sintético (fake).

# Carga y preparación de los datos

Se carga el archivo original y se revisan su estructura, los valores
faltantes, los registros duplicados y las etiquetas disponibles.

In [73]:
import pandas as pd

In [74]:
data = pd.read_csv("data/fake_densenet121.csv")

data.head()

,architecture,transform,audio,fake_probability,prediction,ground_truth
0,densenet121,cqt,1.wav,0.08,real,fake
1,densenet121,dwt,1.wav,0.51,real,fake
2,densenet121,log,1.wav,10.73,real,fake
3,densenet121,mel,1.wav,0.02,real,fake
4,densenet121,cqt,10.wav,2.65,real,fake


In [75]:
print("Filas y columnas:", data.shape)

data.info()

Filas y columnas: (2568, 6)
<class 'pandas.DataFrame'>
RangeIndex: 2568 entries, 0 to 2567
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   architecture      2568 non-null   str    
 1   transform         2568 non-null   str    
 2   audio             2568 non-null   str    
 3   fake_probability  2568 non-null   float64
 4   prediction        2568 non-null   str    
 5   ground_truth      2568 non-null   str    
dtypes: float64(1), str(5)
memory usage: 120.5 KB


In [76]:
# Valores faltantes por columna.
data.isna().sum()

architecture        0
transform           0
audio               0
fake_probability    0
prediction          0
ground_truth        0
dtype: int64

In [77]:
# Registros repetidos para un mismo audio y representación.
data.duplicated(subset=["audio", "transform"]).sum()

np.int64(0)

In [78]:
# Etiquetas verdaderas disponibles.
data["ground_truth"].value_counts()

ground_truth
fake    2568
Name: count, dtype: int64

In [79]:
# Cantidad de registros por representación.
data["transform"].value_counts()

transform
cqt    642
dwt    642
log    642
mel    642
Name: count, dtype: int64

In [80]:
# Rango de las puntuaciones.
data["fake_probability"].agg(["min", "max"])

min      0.0
max    100.0
Name: fake_probability, dtype: float64

In [81]:
# Comprobar las representaciones disponibles.
representations = ["log", "mel", "dwt", "cqt"]

assert set(data["transform"]) == set(representations)

# Cada audio debe tener cuatro registros y cuatro representaciones.
assert data.groupby("audio").size().eq(4).all()

assert data.groupby("audio")["transform"].nunique().eq(4).all()

# Cada audio debe tener una sola etiqueta verdadera.
assert data.groupby("audio")["ground_truth"].nunique().eq(1).all()

print("Estructura de los audios verificada.")

Estructura de los audios verificada.


# Organización de los datos

El archivo original contiene cuatro registros por audio.
Se reorganizan las puntuaciones para obtener una sola fila por audio,
con las columnas LOG, MEL, DWT y CQT y su etiqueta verdadera.

In [82]:
audio_table = data.pivot(
    index="audio",
    columns="transform",
    values="fake_probability"
)

audio_table = audio_table[representations]
audio_table.columns = ["LOG", "MEL", "DWT", "CQT"]

audio_table.head()

,LOG,MEL,DWT,CQT
audio,,,,
1.wav,10.73,0.02,0.51,0.08
10.wav,99.96,0.05,0.74,2.65
100.wav,6.76,0.01,8.18,8.97
101.wav,0.51,0.00,2.38,5.49
102.wav,0.03,0.19,2.39,0.10


In [83]:
labels = data.groupby("audio")["ground_truth"].first()

audio_table["ground_truth"] = labels
audio_table = audio_table.reset_index()
audio_table.columns.name = None

audio_table.head()

,audio,LOG,MEL,DWT,CQT,ground_truth
0,1.wav,10.73,0.02,0.51,0.08,fake
1,10.wav,99.96,0.05,0.74,2.65,fake
2,100.wav,6.76,0.01,8.18,8.97,fake
3,101.wav,0.51,0.00,2.38,5.49,fake
4,102.wav,0.03,0.19,2.39,0.10,fake


In [84]:
audio_table["target"] = audio_table["ground_truth"].map({
    "real": 0,
    "fake": 1
})

audio_table.head()

,audio,LOG,MEL,DWT,CQT,ground_truth,target
0,1.wav,10.73,0.02,0.51,0.08,fake,1
1,10.wav,99.96,0.05,0.74,2.65,fake,1
2,100.wav,6.76,0.01,8.18,8.97,fake,1
3,101.wav,0.51,0.00,2.38,5.49,fake,1
4,102.wav,0.03,0.19,2.39,0.10,fake,1


In [85]:
audio_table["audio_number"] = (
    audio_table["audio"]
    .str.replace(".wav", "", regex=False)
    .astype(int)
)

audio_table = audio_table.sort_values("audio_number")
audio_table = audio_table.reset_index(drop=True)

print("Total de audios:", len(audio_table))

audio_table.head()

Total de audios: 642


,audio,LOG,MEL,DWT,CQT,ground_truth,target,audio_number
0,1.wav,10.73,0.02,0.51,0.08,fake,1,1
1,2.wav,84.34,0.56,17.48,0.67,fake,1,2
2,3.wav,63.58,0.15,1.33,14.12,fake,1,3
3,4.wav,88.24,11.55,1.19,1.72,fake,1,4
4,5.wav,26.95,0.06,1.51,2.11,fake,1,5


# División de los datos

Se aplica la partición indicada en la guía:

- Entrenamiento: audios numerados del 1 al 450.
- Validación: audios numerados desde el 451.

El conjunto de validación se reserva para evaluar la solución final.
No se utilizará para seleccionar modelos, parámetros ni umbrales.

In [86]:
# División fija indicada en la guía.
train_data = audio_table[
    audio_table["audio_number"].between(1, 450)
].copy()

validation_data = audio_table[
    audio_table["audio_number"] >= 451
].copy()

assert len(train_data) + len(validation_data) == len(audio_table)

print("Audios de entrenamiento:", len(train_data))
print("Audios de validación:", len(validation_data))

Audios de entrenamiento: 450
Audios de validación: 192


In [87]:
print("Clases en entrenamiento:")
print(train_data["ground_truth"].value_counts())

print("\nClases en validación:")
print(validation_data["ground_truth"].value_counts())

Clases en entrenamiento:
ground_truth
fake    450
Name: count, dtype: int64

Clases en validación:
ground_truth
fake    192
Name: count, dtype: int64


In [88]:
features = ["LOG", "MEL", "DWT", "CQT"]

X_train = train_data[features]
y_train = train_data["target"]

X_val = validation_data[features]
y_val = validation_data["target"]

print("Entradas de entrenamiento:", X_train.shape)
print("Entradas de validación:", X_val.shape)

if y_train.nunique() < 2 or y_val.nunique() < 2:
    print(
        "Pendiente: incorporar y revisar los datos naturales. "
        "La evaluación de ambas clases todavía no es posible."
    )

Entradas de entrenamiento: (450, 4)
Entradas de validación: (192, 4)
Pendiente: incorporar y revisar los datos naturales. La evaluación de ambas clases todavía no es posible.


# Evaluación de modelos individuales

# Método de fusión

# Evaluación y conclusiones